# Markdown Hierarchy Parser

> Parse Markdown into heading-addressable sections while preserving each section's source text

In [ ]:
#| default_exp md_hier

In [ ]:
#| export
import re
from fastcore.utils import *

In [ ]:
from fastcore.test import *

Markdown documents already divide themselves into useful semantic units. `HeadingDict` preserves that hierarchy, so a long document can be inspected by section instead of searched and sliced as one flat string. Every node is a dictionary of child headings with the complete Markdown for that section in `.text`.

## Heading trees

`HeadingDict` supports normal dictionary lookup. `at()` follows a heading path in one call, `paths()` lists every available path, and `find()` returns a heading when its title occurs exactly once. Explicit paths are preferable when a title such as "Hooks" appears in more than one place.

In [ ]:
#| export
class HeadingDict(dict):
    "A dictionary of child headings with the section Markdown in `text`."
    def __init__(self, text='', *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.text = text

    def paths(self):
        "List every heading path as a tuple."
        res = []
        for title,node in self.items():
            res.append((title,))
            res += [(title,*path) for path in node.paths()]
        return res

    def at(self, *path):
        "Return the node at `path`."
        node = self
        for title in path: node = node[title]
        return node

    def find(self, title):
        "Return the node whose heading is uniquely named `title`."
        paths = [path for path in self.paths() if path[-1] == title]
        if len(paths) != 1: raise KeyError(f"Expected one heading named {title!r}, found {len(paths)}: {paths}")
        return self.at(*paths[0])

<div class="prose" markdown="1">

---

### create_heading_dict

```python
def create_heading_dict(
    text, rm_fenced:bool=True
):
```

*Create a nested dictionary structure from markdown headings.*

</div>

In [ ]:
#| export
def create_heading_dict(text, rm_fenced=True):
    "Create a nested `HeadingDict` from Markdown headings."
    lines = text.splitlines()
    headings,fence = [],None
    for i,line in enumerate(lines):
        if match := re.match(r'^\s*(`{3,}|~{3,})', line):
            marker = match[1][0]
            if fence == marker: fence = None
            elif fence is None: fence = marker
            if rm_fenced: continue
        if fence and rm_fenced: continue
        if match := re.match(r'^(#{1,6})\s+(.+?)\s*#*\s*$', line):
            headings.append(dict(level=len(match[1]), title=match[2], line=i))

    for i,heading in enumerate(headings):
        end = next((h['line'] for h in headings[i+1:] if h['level'] <= heading['level']), len(lines))
        heading['text'] = '\n'.join(lines[heading['line']:end]).strip()

    result = HeadingDict(text)
    stack,levels = [result],[0]
    for heading in headings:
        while len(stack) > 1 and levels[-1] >= heading['level']:
            stack.pop()
            levels.pop()
        parent,title = stack[-1],heading['title']
        if title in parent: raise ValueError(f"Duplicate sibling heading {title!r}")
        parent[title] = node = HeadingDict(heading['text'])
        stack.append(node)
        levels.append(heading['level'])
    return result

In [ ]:
sample_md = r'''# Hooks

Shared hook behavior.

## Common input fields

Every hook receives `session_id`.

## Hooks

### SessionStart

Runs at thread start.

```python
# This is code, not a heading
```

### PostCompact

Runs after compaction.'''

result = create_heading_dict(sample_md)
result.paths()

Available sections:
  Introduction
  Appendix

Root document has 328 characters of text


Use `at()` when you know the path. The returned node includes its heading and all Markdown beneath it, up to the next heading at the same or a higher level.

In [ ]:
session = result.at('Hooks', 'Hooks', 'SessionStart')
session.text

### Installation

Run the following command:

```bash
# Install the packackge
pip install our-package
```


`find()` is shorter when a title is unique anywhere in the document. It raises when the title is absent or ambiguous, so it cannot quietly select the wrong section.

In [ ]:
test_eq(result.find('SessionStart'), session)
test_fail(lambda: result.find('Hooks'), contains='found 2')

## Getting Started

Follow these steps to begin.

### Installation

Run the following command:

```bash
# Install the packackge
pip install our-package
```

### Configuration

Set up your config file.…


## Fenced code and duplicate headings

Markdown examples often contain lines that look like headings. Backtick and tilde fenced blocks are ignored by default, so `# This is code` does not appear in the heading tree.

In [ ]:
expected = r'''### SessionStart

Runs at thread start.

```python
# This is code, not a heading
```'''
test_eq(result.find('SessionStart').text, expected)
test_eq([path for path in result.paths() if path[-1] == 'This is code'], [])

Structure:
Root keys: ['Introduction', 'Appendix']
Introduction subkeys: ['Getting Started', 'Advanced Usage']
Getting Started subkeys: ['Installation', 'Configuration']

Type of result: <class 'toolslm.md_hier.HeadingDict'>
Type of subsection: <class 'toolslm.md_hier.HeadingDict'>
Has text attribute: True


### Duplicate siblings

A dictionary cannot represent two children with the same key. Rather than silently discard the first section, `create_heading_dict` raises for duplicate sibling headings. Repeated titles under different parents are fine and remain addressable by path.

In [ ]:
test_fail(lambda: create_heading_dict('# A\n## Same\n## Same'), contains='Duplicate sibling heading')
test_eq(result.paths()[-1], ('Hooks', 'Hooks', 'PostCompact'))